# Логистическая регрессия и вероятностный ответ

Автор: Чернуха Алексей

CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Хороший спам-фильтр не просто объявляет письмо «спамом» или «не спамом». Он оценивает, насколько уверен: «это спам с вероятностью 95%», «а это — всего с вероятностью 55%». Такая уверенность полезна: первое письмо можно сразу убрать, а второе оставить во входящих и пометить как подозрительное.

После занятия вы сможете:

- Посчитать вероятность класса по признакам объекта с помощью логистической регрессии.
- Объяснить, что такое порог и как он меняет число ошибок модели.
- Посчитать accuracy, precision и recall и понять, чем они отличаются.
- Сравнить логистическую регрессию и решающее дерево на одних и тех же данных.

В этом занятии сначала на одном письме вручную пройдём путь от признаков до вероятности. Потом посмотрим, как вероятность превращается в решение и как измеряют его качество. В конце проверим всё на задаче, где ошибки особенно дороги: по снимку клеток определим, злокачественна ли опухоль.

## От признаков письма к вероятности

Пусть фильтр смотрит на два признака письма:

- $x_1$ — сколько ссылок в тексте;
- $x_2$ — знаком ли отправитель: 1, если он есть в контактах, и 0, если нет.

Ответ тоже запишем числом: спам — $y=1$, обычное письмо — $y=0$. Это два класса.

Идея простая: каждый признак вносит свой вклад в «подозрительность» письма, а вклады складываются. Вклад признака — это сам признак, умноженный на его **вес**. К сумме добавляют ещё **свободный член** $b$ — число, с которого начинается счёт. Получается **оценка** письма:

$$z=b+w_1x_1+w_2x_2.$$

Допустим, обучение дало такие числа: $b=-2$, $w_1=1{,}2$, $w_2=-1{,}5$. Тогда

$$z=-2+1{,}2\cdot x_1-1{,}5\cdot x_2.$$

Каждое слагаемое читается отдельно:

- $-2$ — стартовое значение: письмо без ссылок от незнакомого отправителя выглядит скорее обычным;
- $+1{,}2\cdot x_1$ — каждая ссылка добавляет 1,2 к подозрительности;
- $-1{,}5\cdot x_2$ — знакомый отправитель убавляет её на 1,5.

Возьмём письмо с тремя ссылками от незнакомого отправителя: $x_1=3$, $x_2=0$. Подставляем:

$$z=-2+1{,}2\cdot3-1{,}5\cdot0=-2+3{,}6=1{,}6.$$

Оценка 1,6 — ещё не вероятность. Вероятность лежит между 0 и 1, а оценка может быть любым числом: и −50, и 700. Нужен способ уложить любое число в промежуток от 0 до 1.

Такой способ называется **сигмоидой**:

$$p=\sigma(z)=\frac{1}{1+e^{-z}}.$$



Сигмоида берёт любое число и сжимает его в промежуток от 0 до 1. Чем больше $z$, тем ближе $p$ к 1; чем сильнее $z$ отрицательно, тем ближе $p$ к 0; при $z=0$ получается ровно 0,5. Поэтому результат можно читать как вероятность спама.

Посчитаем для нашего письма. Число $e\approx2{,}718$ — математическая константа, а $e^{x}$ есть кнопкой на калькуляторе:

$$e^{-1{,}6}\approx0{,}202,\qquad
p=\frac{1}{1+0{,}202}=\frac{1}{1{,}202}\approx0{,}832.$$

Модель оценила вероятность спама примерно в **83%**.

Соберём все шаги в одну формулу. Модель берёт признаки письма, считает оценку и пропускает её через сигмоиду:

$$p=\sigma(w_1x_1+w_2x_2+b)=\frac{1}{1+e^{-(w_1x_1+w_2x_2+b)}}.$$

Это и есть **логистическая регрессия**. Её ответ $p$ — вероятность класса 1, то есть спама. Слово «регрессия» здесь не про предсказание произвольных чисел: модель выдаёт вероятность, а класс письма определяется по ней отдельным шагом. Поэтому логистическая регрессия решает задачу классификации.

## Откуда берутся веса

Числа $b=-2$, $w_1=1{,}2$, $w_2=-1{,}5$ мы просто назвали. На самом деле модель подбирает их сама по письмам с известными ответами. Для этого нужна мера ошибки: чем увереннее модель ошибается, тем больше штраф.

Для одного письма с настоящим ответом $y$ и вероятностью спама $p$ штраф такой:

$$\ell=-\bigl(y\ln p+(1-y)\ln(1-p)\bigr).$$

Здесь $\ln$ — натуральный логарифм (кнопка $\ln$ на калькуляторе). Формула выглядит сложнее, чем есть. Для спама ($y=1$) второе слагаемое обнуляется, и остаётся $\ell=-\ln p$. Для обычного письма ($y=0$) остаётся $\ell=-\ln(1-p)$. В обоих случаях штраф — это $-\ln$ от вероятности, которую модель дала правильному ответу: чем она меньше, тем штраф больше.

Спаму с $p=0{,}9$ достанется $-\ln0{,}9\approx0{,}105$, а спаму с $p=0{,}1$ — уже $-\ln0{,}1\approx2{,}303$: уверенная ошибка стоит дорого.

Средний штраф по всем обучающим письмам называют **логарифмической ошибкой** $L$. Посчитаем её для двух писем: спама с тремя ссылками от незнакомого ($p\approx0{,}832$) и обычного письма с одной ссылкой от знакомого ($z=-2+1{,}2-1{,}5=-2{,}3$, $p\approx0{,}091$):

$$L=\frac{-\ln0{,}832-\ln(1-0{,}091)}{2}\approx\frac{0{,}184+0{,}096}{2}=0{,}140.$$

Веса подбирают маленькими шагами: каждый раз немного сдвигают их так, чтобы $L$ уменьшилась. Это **градиентный спуск**. Направление сдвига подсказывает производная; для логистической регрессии она получается простой:

$$\frac{\partial L}{\partial w_1}=\frac1n\sum(p-y)\,x_1.$$

Здесь $n$ — число писем. Разность $p-y$ показывает промах модели на письме, а множитель $x_1$ — вклад ссылок в этот промах. Подставим наши два письма:

$$\frac{\partial L}{\partial w_1}=\frac{(0{,}832-1)\cdot3+(0{,}091-0)\cdot1}{2}=\frac{-0{,}504+0{,}091}{2}\approx-0{,}206.$$

Производная отрицательная, значит, $w_1$ нужно увеличить. Размер шага задаёт **скорость обучения** $\eta$; возьмём $\eta=0{,}5$:

$$w_1\leftarrow1{,}2-0{,}5\cdot(-0{,}206)\approx1{,}303.$$

Вес ссылок вырос: спаму с тремя ссылками модель дала слишком малую вероятность. В библиотеке сотни таких шагов выполняет метод `fit`.

## Как оценить ответы: accuracy, precision, recall

Чтобы получить окончательный ответ, нужно решить, с какой вероятности письмо считать спамом. Начнём с самого очевидного: если $p\ge0{,}5$, письмо уходит в спам. Наше письмо с $p\approx0{,}832$ попадёт в спам.

Возьмём шесть писем, для которых модель уже посчитала вероятность спама. В третьем столбце — настоящий класс, в четвёртом — ответ модели при $p\ge0{,}5$, в пятом — группа, в которую попало письмо.

| Письмо | $p$ | Настоящий класс | Ответ модели | Группа |
| --- | --- | --- | --- | --- |
| А | 0,10 | обычное | обычное | TN |
| Б | 0,35 | спам | обычное | FN |
| В | 0,45 | обычное | обычное | TN |
| Г | 0,60 | спам | спам | TP |
| Д | 0,75 | обычное | спам | FP |
| Е | 0,90 | спам | спам | TP |

Здесь **TP** — спам правильно отправили в спам, **TN** — обычное письмо правильно оставили во входящих. Ошибки двух видов: **FP**, ложное срабатывание, — обычное письмо ушло в спам; **FN**, пропуск, — спам остался во входящих.

Число писем в группах даёт три метрики качества:

- **accuracy** — доля всех верных ответов:
  $$\text{accuracy}=\frac{TP+TN}{TP+TN+FP+FN}=\frac{2+2}{6}\approx0{,}67;$$
- **precision** (точность) — какая доля писем в папке «спам» действительно спам, то есть можно ли ей доверять:
  $$\text{precision}=\frac{TP}{TP+FP}=\frac{2}{2+1}\approx0{,}67;$$
- **recall** (полнота) — какую долю всего спама мы поймали, то есть много ли упустили:
  $$\text{recall}=\frac{TP}{TP+FN}=\frac{2}{2+1}\approx0{,}67.$$

## Порог двигает метрики

Число 0,5 мы взяли просто как самое очевидное, но обязательным оно не является. Введём **порог** $t$: письмо уходит в спам, если $p\ge t$. Вероятности при этом не меняются — меняется только правило, по которому мы превращаем их в ответ.



Снизим порог до 0,3: в спам добавятся Б и В. Спам Б мы теперь поймали, но обычное письмо В ошибочно убрали. При пороге 0,8 срабатывает только Е.

| Порог | TP | FP | FN | TN | accuracy | precision | recall |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0,3 | 3 | 2 | 0 | 1 | $4/6\approx0{,}67$ | $3/5=0{,}6$ | $3/3=1$ |
| 0,5 | 2 | 1 | 1 | 2 | $4/6\approx0{,}67$ | $2/3\approx0{,}67$ | $2/3\approx0{,}67$ |
| 0,8 | 1 | 0 | 2 | 3 | $4/6\approx0{,}67$ | $1/1=1$ | $1/3\approx0{,}33$ |

Accuracy на всех трёх порогах одинаковая, а precision и recall движутся в разные стороны. Низкий порог ловит весь спам, но в папку попадает лишнее: recall растёт, precision падает. Высокий порог делает папку «спам» чистой, зато пропускает больше спама. Порог выбирают под задачу: что хуже — потерять важное письмо или увидеть лишнюю рекламу. Это ручка, которой можно двигать метрики модели, не переобучая её.

## Практика: определяем опухоль по снимку клеток

Проверим всё на реальной задаче, где многое зависит от порога. Набор **breast_cancer** из scikit-learn — это 569 опухолей. Для каждой измерены 30 признаков по снимку клеток: радиус, площадь, гладкость контура и другие. Нужно определить, злокачественна ли опухоль (класс 1, таких 212) или доброкачественна (класс 0).

Ошибки здесь неравноценны. Пропустить злокачественную опухоль (FN) гораздо опаснее, чем отправить здорового человека на лишнее обследование (FP). Поэтому порог выберем не по привычке, а по цене ошибок.

Сравним логистическую регрессию и решающее дерево глубины 4. Площадь измеряется сотнями, а гладкость — сотыми долями, поэтому для логистической регрессии приведём признаки к одной шкале с помощью `StandardScaler`. Функция `make_pipeline` соединяет масштабирование и модель в одну цепочку, и масштаб подбирается только по обучающей части.

Данные делим один раз: 60% — на обучение, 20% — для выбора порога, 20% — для итоговой проверки. `stratify=y` сохраняет долю злокачественных опухолей в каждой части.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)  # 1 — злокачественная


In [ ]:
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_rest, y_rest, test_size=0.25, random_state=42, stratify=y_rest
)


In [ ]:
models = {
    "Логистическая регрессия": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
    "Дерево": DecisionTreeClassifier(max_depth=4, random_state=42),
}
for model in models.values():
    model.fit(X_train, y_train)


Функция `report` по вероятностям и порогу считает все метрики сразу:

In [ ]:
def report(y_true, probabilities, threshold):
    predictions = probabilities >= threshold
    tp = int((predictions & (y_true == 1)).sum())
    fp = int((predictions & (y_true == 0)).sum())
    fn = int((~predictions & (y_true == 1)).sum())
    return {"FP": fp, "FN": fn,
            "accuracy": round(1 - (fp + fn) / len(y_true), 3),
            "precision": round(tp / (tp + fp), 3) if tp + fp else 0.0,
            "recall": round(tp / (tp + fn), 3),
            "цена": 4 * fn + fp}


### Выбираем порог на проверочной части

Метод `predict_proba` возвращает по столбцу на каждый класс; столбец с номером 1 — это вероятность класса 1.

Договоримся, что пропуск в четыре раза хуже ложной тревоги. Тогда цена ошибок равна $C=4\,\mathrm{FN}+\mathrm{FP}$. Для каждой модели сравним три порога — 0,2; 0,5 и 0,8 — и выберем тот, при котором цена на проверочной части минимальна.

In [ ]:
chosen = {}
for name, model in models.items():
    probabilities = model.predict_proba(X_valid)[:, 1]
    costs = {}
    for threshold in [0.2, 0.5, 0.8]:
        row = report(y_valid, probabilities, threshold)
        costs[threshold] = row["цена"]
        print(name, threshold, row)
    chosen[name] = min(costs, key=costs.get)


У логистической регрессии на проверочной части при всех трёх порогах ровно один пропуск, поэтому выбор решают ложные тревоги: побеждает самый строгий порог 0,8. Дереву лучше всего подходит порог 0,2.

### Итоговая проверка

In [ ]:
for name, model in models.items():
    probabilities = model.predict_proba(X_test)[:, 1]
    print(name, "порог:", chosen[name],
          report(y_test, probabilities, chosen[name]))


| Модель | Порог | FP | FN | accuracy | precision | recall | Цена |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Логистическая регрессия | 0,8 | 0 | 5 | 0,956 | 1,000 | 0,881 | $4\cdot5+0=20$ |
| Дерево | 0,2 | 4 | 6 | 0,912 | 0,900 | 0,857 | $4\cdot6+4=28$ |

Логистическая регрессия лучше по всем метрикам. Но на тесте порог 0,8 пропустил 5 опухолей из 42, хотя на проверочной части был всего один пропуск: проверочная часть небольшая, и порог по ней получился слишком строгим. Подправлять порог по тесту теперь нельзя: тест нужен как независимая проверка, а подгонка под него сделает оценку нечестной.

**Самостоятельно.** Правило «всегда доброкачественная» ни разу не срабатывает. Посчитайте для него на тесте accuracy и recall. Затем на проверочной части найдите для логистической регрессии наименьший порог из `[0.05, 0.1, ..., 0.95]`, при котором precision не ниже 0,95.



## Самопроверка

### 1. Логистическая регрессия считает оценку $z=b+w_1x_1+w_2x_2$ и применяет к ней сигмоиду. Зачем?

A. Увеличить число признаков.<br>
B. Получить число между 0 и 1.<br>
C. Сразу выбрать порог.<br>
D. Заменить обучающие ответы.

<details>
<summary>Подсказка</summary>

Линейная сумма может равняться и −5, и 12.

</details>



### 2. Чему равна $\sigma(z)=\frac{1}{1+e^{-z}}$ при $z=0$?

A. 0.<br>
B. 1.<br>
C. 0,5.<br>
D. Зависит от порога.

<details>
<summary>Подсказка</summary>

Подставьте $e^0=1$ в формулу.

</details>



### 3. Оценка письма $z=-2+1{,}2x_1-1{,}5x_2$, где $x_1$ — число ссылок, $x_2=1$ для знакомого отправителя. Чему равна $z$ у письма с одной ссылкой от знакомого?

A. −2,3.<br>
B. 0,7.<br>
C. 1,6.<br>
D. 4,7.

<details>
<summary>Подсказка</summary>

Подставьте $x_1=1$ и $x_2=1$.

</details>



### 4. Модель относит объект к классу 1, если $p\ge t$. Она выдала $p=0{,}4$, порог $t=0{,}3$. Какой класс получим?

A. Класс 0, потому что $p<0{,}5$.<br>
B. Класс 1, потому что $p\ge0{,}3$.<br>
C. Оба класса.<br>
D. Нужны настоящие ответы.

<details>
<summary>Подсказка</summary>

Используйте заданный порог.

</details>



### 5. Спам-фильтр отправил в спам обычное письмо. Какая это ошибка?

A. Пропуск FN.<br>
B. Ложное срабатывание FP.<br>
C. Верный ответ.<br>
D. Ошибка разбиения данных.

<details>
<summary>Подсказка</summary>

Модель сработала на объекте, где класса 1 на самом деле нет.

</details>



### 6. Письмо уходит в спам, если $p\ge t$. Порог $t$ повышаем, вероятности не меняются. Что будет с FP и FN?

A. FP не увеличится, FN не уменьшится.<br>
B. FP не уменьшится, FN не увеличится.<br>
C. Обе ошибки обязательно уменьшатся.<br>
D. Вероятности изменятся.

<details>
<summary>Подсказка</summary>

При более высоком пороге меньше объектов получают класс 1.

</details>



### 7. Для спама ошибка равна $-\ln p$. Модель дала письму $p=0{,}1$ вместо $p=0{,}9$. Что станет с ошибкой?

A. Уменьшится.<br>
B. Не изменится.<br>
C. Увеличится.<br>
D. Всегда будет нулевой.

<details>
<summary>Подсказка</summary>

Для настоящего спама используем $-\ln p$.

</details>



### 8. Данные делят на обучающую, проверочную и тестовую части. На какой выбирают порог?

A. На обучающей части.<br>
B. На проверочной части.<br>
C. На тестовой части.<br>
D. На всех данных сразу.

<details>
<summary>Подсказка</summary>

Одна часть обучает веса, другая выбирает настройку, третья проверяет готовое решение.

</details>



### 9. При пороге 0,5 у модели TP = 6, FP = 2, FN = 3. Чему равны precision и recall?

A. 0,75 и 0,67.<br>
B. 0,67 и 0,75.<br>
C. 0,8 и 0,6.<br>
D. 0,6 и 0,8.

<details>
<summary>Подсказка</summary>

Precision делит TP на все срабатывания, recall — на все настоящие объекты класса 1.

</details>



### 10. Правило «всегда доброкачественная» на тесте из 114 опухолей (42 злокачественные) даёт accuracy $72/114\approx63\%$. Почему этого мало?

A. Accuracy нельзя считать для двух классов.<br>
B. Его recall равен 0: оно не находит ни одной злокачественной опухоли.<br>
C. Его precision равен 1.<br>
D. Правило использует тестовую часть.

<details>
<summary>Подсказка</summary>

Сколько злокачественных опухолей это правило отметит?

</details>




## Решения задач

### Задача 1

В тесте 114 опухолей, из них 42 злокачественных. Правило «всегда 0» даёт TP = 0 и FN = 42: accuracy $=72/114\approx0{,}632$, recall $=0/42=0$. Высокая доля верных ответов не значит, что модель находит нужный класс.

In [ ]:
always_zero = np.zeros(len(y_test))
print(report(y_test, always_zero, 0.5))

probabilities = models["Логистическая регрессия"].predict_proba(X_valid)[:, 1]
for threshold in [round(0.05 * k, 2) for k in range(1, 20)]:
    if report(y_valid, probabilities, threshold)["precision"] >= 0.95:
        print("порог:", threshold, report(y_valid, probabilities, threshold))
        break


Получится порог 0,35: на проверочной части FP = 2, FN = 1, precision $\approx0{,}955$, recall $\approx0{,}977$.

### Задача 2

**B.** Сигмоида превращает оценку в вероятность класса 1. Порог применяют следующим шагом.

### Задача 3

**C.** $p=1/(1+1)=0{,}5$. Порог не участвует в вычислении вероятности.

### Задача 4

**A.** $-2+1{,}2-1{,}5=-2{,}3$. Соответствующая вероятность примерно 0,091, меньше 0,5.

### Задача 5

**B.** Сравниваем с 0,3: $0{,}4\ge0{,}3$. Настоящий ответ нужен для оценки ошибки, а не для прогноза.

### Задача 6

**B.** Предсказан класс 1 при настоящем классе 0. Пропуск — обратный случай: настоящий спам оставили во входящих.

### Задача 7

**A.** Срабатываний становится меньше или столько же. Часть ложных срабатываний может исчезнуть, часть настоящих положительных ответов — превратиться в пропуски.

### Задача 8

**C.** Ошибка вырастет примерно с 0,105 до 2,303. Модель уверенно назначила малую вероятность настоящему классу.

### Задача 9

**B.** Порог — настройка решения. Тестовая часть остаётся для итоговой оценки после выбора.

### Задача 10

**A.** precision $=6/(6+2)=0{,}75$, recall $=6/(6+3)\approx0{,}67$. Вариант B путает формулы местами.

### Задача 11

**B.** Большинство опухолей доброкачественные, поэтому доля верных ответов высокая, но TP = 0 и recall $=0$. Поэтому вместе с accuracy смотрят на recall и precision.